In [6]:
import random
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from sklearn.metrics import cohen_kappa_score

from src.config import IMG_DIR, SPLITS_DIR, PROCESSED_DIR
from src.data.dataset import DiabeticRetinopathyDataset, get_dataloader

In [7]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

In [8]:
train_df = pd.read_csv(SPLITS_DIR / "train_split.csv")
val_df = pd.read_csv(SPLITS_DIR / "val_split.csv")

train_dataset = DiabeticRetinopathyDataset(
    img_dir=str(IMG_DIR),
    annotations_df=train_df,
    split="train",
    preprocessed_dir=PROCESSED_DIR / "images_processed"
)
val_dataset = DiabeticRetinopathyDataset(
    img_dir=str(IMG_DIR),
    annotations_df=val_df,
    split="val",
    preprocessed_dir=PROCESSED_DIR / "images_processed"
)

train_loader = get_dataloader(train_dataset, batch_size=32, num_workers=2)
val_loader = get_dataloader(val_dataset, batch_size=32, num_workers=2)

In [9]:
class BaselinePretrainedModel(nn.Module):
    def __init__(self, model_name='efficientnet_lite0', num_classes=5):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

In [10]:
set_seed()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = BaselinePretrainedModel()
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

num_epochs = 10
best_qwk = -1.0

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    total_samples = 0

    for batch in train_loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        
        out = model(images)
        loss = loss_fn(out, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        current_batch_size = images.shape[0]
        total_samples += current_batch_size
        train_loss += loss.item() * current_batch_size
        
    avg_train_loss = train_loss / total_samples

    model.eval()
    val_loss = 0.0
    total_samples = 0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            
            out = model(images)
            loss = loss_fn(out, labels)
            current_batch_size = images.shape[0]
            total_samples += current_batch_size
            val_loss += loss.item() * current_batch_size

            preds = torch.argmax(out, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_samples

    val_qwk = cohen_kappa_score(all_targets, all_preds, weights='quadratic')
    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val QWK: {val_qwk:.4f}")

    if val_qwk > best_qwk:
        best_qwk = val_qwk
        torch.save(model.state_dict(), '/kaggle/working/best_efficientnet_lite0.pth')
        print(f"Saved new best model with QWK: {best_qwk:.4f}")

Device: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B / 18.8MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch [1/10] | Train Loss: 0.9991 | Val Loss: 0.8592 | Val QWK: 0.7131
Saved new best model with QWK: 0.7131
Epoch [2/10] | Train Loss: 0.6853 | Val Loss: 0.8025 | Val QWK: 0.7376
Saved new best model with QWK: 0.7376
Epoch [3/10] | Train Loss: 0.5914 | Val Loss: 0.7311 | Val QWK: 0.7685
Saved new best model with QWK: 0.7685
Epoch [4/10] | Train Loss: 0.5261 | Val Loss: 0.7214 | Val QWK: 0.8030
Saved new best model with QWK: 0.8030
Epoch [5/10] | Train Loss: 0.4775 | Val Loss: 0.6335 | Val QWK: 0.8079
Saved new best model with QWK: 0.8079
Epoch [6/10] | Train Loss: 0.4661 | Val Loss: 0.6363 | Val QWK: 0.8095
Saved new best model with QWK: 0.8095
Epoch [7/10] | Train Loss: 0.4094 | Val Loss: 0.7444 | Val QWK: 0.7898
Epoch [8/10] | Train Loss: 0.3865 | Val Loss: 0.6318 | Val QWK: 0.8285
Saved new best model with QWK: 0.8285
Epoch [9/10] | Train Loss: 0.3554 | Val Loss: 0.6145 | Val QWK: 0.8407
Saved new best model with QWK: 0.8407
Epoch [10/10] | Train Loss: 0.3317 | Val Loss: 0.6566 | V